# Train Hybrid Music Recommender on Your Dataset

Put your dataset CSVs inside `data/my_dataset/` or change `DATA_DIR` below.

Required files:

- `interactions.csv`: `user_id, song_id, play_count`
- `user_features.csv`: `user_id` plus user feature columns
- `song_characteristics.csv`: `song_id` plus song metadata columns
- `lyric_embeddings.csv`: `song_id` plus numeric embedding columns, or one `embedding` column containing a list like `[0.1, 0.2, ...]`

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "my_dataset"   # change this if needed
OUTPUT_DIR = PROJECT_DIR / "artifacts" / "my_dataset_run"

print("Project:", PROJECT_DIR)
print("Dataset:", DATA_DIR)
print("Outputs:", OUTPUT_DIR)

Project: /Users/suvratsmacbookair/MyLocalFiles/LyricLens/notebooks
Dataset: /Users/suvratsmacbookair/MyLocalFiles/LyricLens/notebooks/data/my_dataset
Outputs: /Users/suvratsmacbookair/MyLocalFiles/LyricLens/notebooks/artifacts/my_dataset_run


## 1. Check Dataset Files

In [12]:
from pathlib import Path

# ---------------------------------------------------------
# Base project directory
# ---------------------------------------------------------

PROJECT_DIR = Path("/Users/suvratsmacbookair/MyLocalFiles/LyricLens"
)

DATA_DIR = PROJECT_DIR / "data" / "large_hybrid"


# ---------------------------------------------------------
# Required dataset files
# ---------------------------------------------------------

required_files = [
    "/Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/interactions.csv",
    "/Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/user_features.csv",
    "/Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/song_characteristics.csv",
    "/Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/lyric_embeddings.csv",
    "/Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/final_songs_FIXED.csv"
]


# ---------------------------------------------------------
# Check whether files exist
# ---------------------------------------------------------

missing_files = []

for name in required_files:
    path = DATA_DIR / name

    if path.exists():
        print(f"✅ {name} OK")
        print(f"   {path}")
    else:
        print(f"❌ {name} MISSING")
        print(f"   Expected at: {path}")
        missing_files.append(name)


# ---------------------------------------------------------
# Stop if anything is missing
# ---------------------------------------------------------

if missing_files:
    raise FileNotFoundError(
        "\n❌ Missing dataset files:\n"
        + "\n".join(f"   - {name}" for name in missing_files)
        + f"\n\nExpected folder:\n{DATA_DIR}"
    )

print("\n✅ All required dataset files found.")

✅ /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/interactions.csv OK
   /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/interactions.csv
✅ /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/user_features.csv OK
   /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/user_features.csv
✅ /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/song_characteristics.csv OK
   /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/song_characteristics.csv
✅ /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/lyric_embeddings.csv OK
   /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/lyric_embeddings.csv
✅ /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/final_songs_FIXED.csv OK
   /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/final_songs_FIXED.csv

✅ All required dataset files found.


In [14]:
for name in required_files:
    df = pd.read_csv(DATA_DIR / name, nrows=5)
    print("\n", name)
    display(df.head())
    print("columns:", list(df.columns))


 /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/interactions.csv


,user_id,song_id,label,is_hard_neg,genre
0,user_0000,25R5Tdr4K7ENb6umHzdCXn,1,False,Pop
1,user_0000,6odIARTda4VMoX6j5VZfLc,0,True,Pop
2,user_0000,1PI5Nlv1bJESN0H1wBa7Pe,0,True,Pop
3,user_0000,2ePgoNrRkl8UKJkdjvMWvw,0,True,Pop
4,user_0000,14lDO5V0Ay0h1t8QTpbfi1,0,True,Pop


columns: ['user_id', 'song_id', 'label', 'is_hard_neg', 'genre']

 /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/user_features.csv


,user_id,affinity_Rock,affinity_Pop,affinity_Electronic,affinity_Folk,affinity_Country,affinity_Hip-Hop,affinity_RandB,affinity_Jazz,affinity_Blues,...,rel_pref_acousticness,rel_pref_instrumentalness,rel_pref_liveness,rel_pref_valence,rel_pref_tempo,avg_popularity_pref,mainstream_score,listening_entropy,diversity_score,n_positives
0,user_0000,0.00,1.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00,...,0.027325,-0.043350,-0.04497,0.064061,-4.140054,20.05,0.2005,0.000000,0.760158,20
1,user_0001,0.00,1.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00,...,-0.078382,-0.041401,-0.00699,0.102591,-2.168754,17.40,0.1740,0.000000,0.752895,20
2,user_0002,0.85,0.0,0.00,0.0,0.0,0.15,0.0,0.0,0.00,...,-0.051773,-0.008880,0.03962,-0.027499,5.767096,20.15,0.2015,0.422709,0.686102,20
3,user_0003,0.75,0.2,0.00,0.0,0.0,0.00,0.0,0.0,0.05,...,-0.065291,-0.083409,0.01934,0.014051,-0.673454,15.31,0.1531,0.687436,0.768468,20
4,user_0004,0.60,0.0,0.35,0.0,0.0,0.00,0.0,0.0,0.00,...,-0.043343,0.035532,-0.03979,-0.101809,18.851246,16.20,0.1620,0.823720,0.717963,20


columns: ['user_id', 'affinity_Rock', 'affinity_Pop', 'affinity_Electronic', 'affinity_Folk', 'affinity_Country', 'affinity_Hip-Hop', 'affinity_RandB', 'affinity_Jazz', 'affinity_Blues', 'affinity_Classical', 'rel_pref_danceability', 'rel_pref_energy', 'rel_pref_loudness', 'rel_pref_speechiness', 'rel_pref_acousticness', 'rel_pref_instrumentalness', 'rel_pref_liveness', 'rel_pref_valence', 'rel_pref_tempo', 'avg_popularity_pref', 'mainstream_score', 'listening_entropy', 'diversity_score', 'n_positives']

 /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/song_characteristics.csv


...

columns: ['song_id', 'name', 'artists', 'album_name', 'genre', 'playlist_subgenre', 'year', 'popularity', 'effective_popularity', 'total_artist_followers', 'avg_artist_popularity', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'dance_energy', 'mood_score', 'tempo_bucket', 'log_artist_followers', 'popularity_tier', 'norm_danceability', 'norm_energy', 'norm_loudness', 'norm_speechiness', 'norm_acousticness', 'norm_instrumentalness', 'norm_liveness', 'norm_valence', 'norm_tempo']

 /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data/lyric_embeddings.csv


,song_id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,7FQyzz3MRRyIHB2xuPjQhL,-0.040392,-0.014218,0.085331,0.029759,0.072617,0.038824,-0.027241,-0.093946,0.085913,...,0.033288,-0.009564,-0.070199,0.056118,-0.020137,0.021022,0.128387,0.045237,-0.033101,-0.028312
1,1bp2PcwZgR9LwBdoERsOuJ,-0.041940,-0.070994,0.125691,0.005189,0.067619,0.002687,0.129885,-0.020481,-0.008081,...,0.085529,-0.010067,0.041193,-0.025716,-0.058122,0.007096,-0.053159,0.011962,-0.060073,-0.022272
2,2AfFuqGQSTBI31kbqE1qo4,-0.000502,-0.028515,0.154366,0.048645,0.016588,-0.057555,0.155483,-0.020181,0.025338,...,-0.007880,-0.027377,-0.038303,0.057789,-0.011123,-0.001249,0.147661,0.067407,-0.107058,-0.084965
3,5qTjvpAfdtyzauTprzgKao,-0.033191,-0.045651,0.072298,0.033687,0.057639,-0.006993,0.098991,0.024701,0.060943,...,0.047501,0.002231,-0.032596,0.076129,-0.011091,-0.017199,0.072418,0.011670,-0.044924,-0.037749
4,6V3SNkvi4BnfmZU0j7s9TQ,-0.074757,0.103775,-0.035076,0.008664,0.011147,0.038274,0.029722,-0.040300,0.005400,...,0.078474,-0.126427,0.003633,0.050448,-0.044652,0.039909,0.057501,-0.094978,0.015915,-0.066279


columns: ['song_id', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7', 'emb_8', 'emb_9', 'emb_10', 'emb_11', 'emb_12', 'emb_13', 'emb_14', 'emb_15', 'emb_16', 'emb_17', 'emb_18', 'emb_19', 'emb_20', 'emb_21', 'emb_22', 'emb_23', 'emb_24', 'emb_25', 'emb_26', 'emb_27', 'emb_28', 'emb_29', 'emb_30', 'emb_31', 'emb_32', 'emb_33', 'emb_34', 'emb_35', 'emb_36', 'emb_37', 'emb_38', 'emb_39', 'emb_40', 'emb_41', 'emb_42', 'emb_43', 'emb_44', 'emb_45', 'emb_46', 'emb_47', 'emb_48', 'emb_49', 'emb_50', 'emb_51', 'emb_52', 'emb_53', 'emb_54', 'emb_55', 'emb_56', 'emb_57', 'emb_58', 'emb_59', 'emb_60', 'emb_61', 'emb_62', 'emb_63', 'emb_64', 'emb_65', 'emb_66', 'emb_67', 'emb_68', 'emb_69', 'emb_70', 'emb_71', 'emb_72', 'emb_73', 'emb_74', 'emb_75', 'emb_76', 'emb_77', 'emb_78', 'emb_79', 'emb_80', 'emb_81', 'emb_82', 'emb_83', 'emb_84', 'emb_85', 'emb_86', 'emb_87', 'emb_88', 'emb_89', 'emb_90', 'emb_91', 'emb_92', 'emb_93', 'emb_94', 'emb_95', 'emb_96', 'emb_97', 'emb_98',

,song_id,song_name,artist,lyrics,genre,emb_0,emb_1,emb_2,emb_3,emb_4,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,7FQyzz3MRRyIHB2xuPjQhL,Twenties,"[""Wolfgang Lohr"", ""Loredana Grimaudo""]",The past is meeting the future\n The future is...,Electronic,-0.040392,-0.014218,0.085331,0.029759,0.072617,...,0.033288,-0.009564,-0.070199,0.056118,-0.020137,0.021022,0.128387,0.045237,-0.033101,-0.028312
1,1bp2PcwZgR9LwBdoERsOuJ,Country Squire,"[""Tyler Childers""]","Well, tonight I'm up in Chillicothe\nDownwind ...",Country,-0.041940,-0.070994,0.125691,0.005189,0.067619,...,0.085529,-0.010067,0.041193,-0.025716,-0.058122,0.007096,-0.053159,0.011962,-0.060073,-0.022272
2,2AfFuqGQSTBI31kbqE1qo4,Nearness Of You,"[""Chris Botti""]",Its not the pale moon that excites me\nThat th...,Jazz,-0.000502,-0.028515,0.154366,0.048645,0.016588,...,-0.007880,-0.027377,-0.038303,0.057789,-0.011123,-0.001249,0.147661,0.067407,-0.107058,-0.084965
3,5qTjvpAfdtyzauTprzgKao,Hold On,"[""All That Remains""]","Don't touch me, I don't wanna feel your skin\n...",Rock,-0.033191,-0.045651,0.072298,0.033687,0.057639,...,0.047501,0.002231,-0.032596,0.076129,-0.011091,-0.017199,0.072418,0.011670,-0.044924,-0.037749
4,6V3SNkvi4BnfmZU0j7s9TQ,Scourge of Iron,"[""Cannibal Corpse""]",In thrall to the evil lord\nA procession of th...,Rock,-0.074757,0.103775,-0.035076,0.008664,0.011147,...,0.078474,-0.126427,0.003633,0.050448,-0.044652,0.039909,0.057501,-0.094978,0.015915,-0.066279


columns: ['song_id', 'song_name', 'artist', 'lyrics', 'genre', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7', 'emb_8', 'emb_9', 'emb_10', 'emb_11', 'emb_12', 'emb_13', 'emb_14', 'emb_15', 'emb_16', 'emb_17', 'emb_18', 'emb_19', 'emb_20', 'emb_21', 'emb_22', 'emb_23', 'emb_24', 'emb_25', 'emb_26', 'emb_27', 'emb_28', 'emb_29', 'emb_30', 'emb_31', 'emb_32', 'emb_33', 'emb_34', 'emb_35', 'emb_36', 'emb_37', 'emb_38', 'emb_39', 'emb_40', 'emb_41', 'emb_42', 'emb_43', 'emb_44', 'emb_45', 'emb_46', 'emb_47', 'emb_48', 'emb_49', 'emb_50', 'emb_51', 'emb_52', 'emb_53', 'emb_54', 'emb_55', 'emb_56', 'emb_57', 'emb_58', 'emb_59', 'emb_60', 'emb_61', 'emb_62', 'emb_63', 'emb_64', 'emb_65', 'emb_66', 'emb_67', 'emb_68', 'emb_69', 'emb_70', 'emb_71', 'emb_72', 'emb_73', 'emb_74', 'emb_75', 'emb_76', 'emb_77', 'emb_78', 'emb_79', 'emb_80', 'emb_81', 'emb_82', 'emb_83', 'emb_84', 'emb_85', 'emb_86', 'emb_87', 'emb_88', 'emb_89', 'emb_90', 'emb_91', 'emb_92', 'emb_93', 'emb_94

## 2. Load and Prepare Data

In [22]:
# =========================================================
# Cell 16 — Load & Prepare Hybrid Dataset
# =========================================================

from pathlib import Path
import sys
import importlib
import pandas as pd


# ---------------------------------------------------------
# 1. Project paths
# ---------------------------------------------------------

PROJECT_DIR = Path(
    "/Users/suvratsmacbookair/MyLocalFiles/LyricLens"
)

DATA_DIR = PROJECT_DIR / "data"
SRC_DIR = PROJECT_DIR / "src"


# ---------------------------------------------------------
# 2. Verify directories
# ---------------------------------------------------------

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"❌ Project directory not found:\n{PROJECT_DIR}"
    )

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"❌ Data directory not found:\n{DATA_DIR}"
    )

print(f"✅ Project directory: {PROJECT_DIR}")
print(f"✅ Data directory:    {DATA_DIR}")
print(f"✅ Source directory:  {SRC_DIR}")


# ---------------------------------------------------------
# 3. Required dataset files
# ---------------------------------------------------------

required_files = [
    "interactions.csv",
    "user_features.csv",
    "song_characteristics.csv",
    "lyric_embeddings.csv",
    "final_songs_FIXED.csv",
]

missing_files = []

for filename in required_files:
    path = DATA_DIR / filename

    if path.exists():
        print(f"✅ {filename}")
    else:
        print(f"❌ {filename} MISSING")
        missing_files.append(filename)

if missing_files:
    raise FileNotFoundError(
        "\n❌ Missing required dataset files:\n"
        + "\n".join(f"   - {f}" for f in missing_files)
    )


# ---------------------------------------------------------
# 4. Locate project modules
# ---------------------------------------------------------

data_loader_file = None
preprocess_file = None
emotion_theme_file = None

# data_loader.py
for path in SRC_DIR.rglob("data_loader.py"):
    if ".venv" not in path.parts:
        data_loader_file = path
        break

# preprocess.py
for path in SRC_DIR.rglob("preprocess.py"):
    if ".venv" not in path.parts:
        preprocess_file = path
        break

# emotion_theme.py
for path in SRC_DIR.rglob("emotion_theme.py"):
    if ".venv" not in path.parts:
        emotion_theme_file = path
        break


if data_loader_file is None:
    raise ModuleNotFoundError(
        f"❌ data_loader.py not found inside {SRC_DIR}"
    )

if preprocess_file is None:
    raise ModuleNotFoundError(
        f"❌ preprocess.py not found inside {SRC_DIR}"
    )

if emotion_theme_file is None:
    raise ModuleNotFoundError(
        f"❌ emotion_theme.py not found inside {SRC_DIR}"
    )


print("\n" + "=" * 70)
print("PROJECT MODULES")
print("=" * 70)

print(f"data_loader.py   → {data_loader_file}")
print(f"preprocess.py    → {preprocess_file}")
print(f"emotion_theme.py → {emotion_theme_file}")


# ---------------------------------------------------------
# 5. Add correct module paths
# ---------------------------------------------------------

module_dirs = [
    SRC_DIR,
    data_loader_file.parent,
    preprocess_file.parent,
    emotion_theme_file.parent,
]

for directory in module_dirs:
    directory = str(directory)

    if directory not in sys.path:
        sys.path.insert(0, directory)


# ---------------------------------------------------------
# 6. Clear previously imported project modules
# ---------------------------------------------------------

for module_name in [
    "data_loader",
    "preprocess",
    "emotion_theme",
]:
    if module_name in sys.modules:
        del sys.modules[module_name]


# ---------------------------------------------------------
# 7. Import correct project modules
# ---------------------------------------------------------

from data_loader import load_hybrid_csvs
from preprocess import prepare_hybrid_data

import data_loader
import preprocess

print("\n✅ Project modules imported successfully.")

print("\nLoaded modules:")
print("data_loader →", data_loader.__file__)
print("preprocess  →", preprocess.__file__)


# ---------------------------------------------------------
# 8. Inspect interactions.csv
# ---------------------------------------------------------

INTERACTIONS_PATH = DATA_DIR / "interactions.csv"

interactions_df = pd.read_csv(
    INTERACTIONS_PATH,
    low_memory=False
)

print("\n" + "=" * 70)
print("INTERACTIONS DATASET")
print("=" * 70)

print(f"Shape: {interactions_df.shape}")
print(f"Columns: {list(interactions_df.columns)}")


# ---------------------------------------------------------
# 9. Normalize interaction column
# ---------------------------------------------------------

# Normalize column names for comparison only
column_lookup = {
    str(col).strip().lower().replace(" ", "_"): col
    for col in interactions_df.columns
}


# Already correct
if "play_count" in column_lookup:

    actual_column = column_lookup["play_count"]

    if actual_column != "play_count":
        interactions_df = interactions_df.rename(
            columns={actual_column: "play_count"}
        )

    print("✅ Found existing play_count column.")


else:

    # Possible names for play count / interaction strength
    candidate_columns = [
        "count",
        "playcount",
        "play_count",
        "plays",
        "num_plays",
        "number_of_plays",
        "listen_count",
        "listen_counts",
        "interaction_count",
        "interaction_strength",
        "weight",
        "frequency",
        "freq",
        "rating",
        "score",
        "value",
        "interaction",
    ]

    detected_column = None

    for candidate in candidate_columns:
        if candidate in column_lookup:
            detected_column = column_lookup[candidate]
            break

    if detected_column is not None:

        print(
            f"⚠️ 'play_count' not found."
        )
        print(
            f"   Using '{detected_column}' as play_count."
        )

        interactions_df = interactions_df.rename(
            columns={
                detected_column: "play_count"
            }
        )

    else:

        # -------------------------------------------------
        # If there is no numeric interaction-strength
        # column, create a binary play_count.
        #
        # Every recorded interaction = 1 play.
        # -------------------------------------------------

        if "interaction" in column_lookup:

            interaction_col = column_lookup["interaction"]

            print(
                f"⚠️ No play-count column found."
            )
            print(
                f"   Converting '{interaction_col}' into "
                f"binary play_count."
            )

            interactions_df["play_count"] = (
                pd.to_numeric(
                    interactions_df[interaction_col],
                    errors="coerce"
                )
                .fillna(1)
            )

        else:

            print(
                "⚠️ No interaction-strength column found."
            )
            print(
                "   Treating every row as one interaction."
            )

            interactions_df["play_count"] = 1.0


# ---------------------------------------------------------
# 10. Verify required interaction columns
# ---------------------------------------------------------

required_interaction_columns = [
    "user_id",
    "song_id",
    "play_count",
]

missing_interaction_columns = [
    col
    for col in required_interaction_columns
    if col not in interactions_df.columns
]

if missing_interaction_columns:

    raise ValueError(
        "\n❌ interactions.csv still does not contain "
        "the required columns:\n"
        + "\n".join(
            f"   - {col}"
            for col in missing_interaction_columns
        )
        + "\n\nActual columns are:\n"
        + "\n".join(
            f"   - {col}"
            for col in interactions_df.columns
        )
    )


# ---------------------------------------------------------
# 11. Clean interaction data
# ---------------------------------------------------------

interactions_df["play_count"] = pd.to_numeric(
    interactions_df["play_count"],
    errors="coerce"
)

interactions_df = interactions_df.dropna(
    subset=["user_id", "song_id"]
).copy()

interactions_df["play_count"] = (
    interactions_df["play_count"]
    .fillna(0.0)
    .clip(lower=0.0)
)


# ---------------------------------------------------------
# 12. Save normalized interactions temporarily
# ---------------------------------------------------------

NORMALIZED_INTERACTIONS_PATH = (
    DATA_DIR / "interactions_normalized.csv"
)

interactions_df.to_csv(
    NORMALIZED_INTERACTIONS_PATH,
    index=False
)

print(
    f"\n✅ Normalized interactions saved:\n"
    f"{NORMALIZED_INTERACTIONS_PATH}"
)

print(
    f"Rows: {len(interactions_df):,}"
)

print(
    f"Unique users: {interactions_df['user_id'].nunique():,}"
)

print(
    f"Unique songs: {interactions_df['song_id'].nunique():,}"
)


# ---------------------------------------------------------
# 13. Load remaining hybrid datasets
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING HYBRID DATA")
print("=" * 70)

# IMPORTANT:
# data_loader expects play_count, so temporarily point it
# to the normalized interaction file.

raw = load_hybrid_csvs(
    data_dir=DATA_DIR,
    interactions_path=NORMALIZED_INTERACTIONS_PATH,
)

print("✅ Raw hybrid data loaded.")


# ---------------------------------------------------------
# 14. Prepare hybrid data
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("PREPARING HYBRID DATA")
print("=" * 70)

prepared = prepare_hybrid_data(
    interactions=raw.interactions,
    user_features=raw.user_features,
    song_characteristics=raw.song_characteristics,
    lyric_embeddings=raw.lyric_embeddings,
    min_play_count=0.0,
    confidence_alpha=1.0,
    validation_ratio=0.2,
    seed=42,
)

print("✅ Hybrid dataset prepared.")


# ---------------------------------------------------------
# 15. Final summary
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("HYBRID DATASET SUMMARY")
print("=" * 70)

print(
    f"Users                  : "
    f"{len(prepared.index_to_user_id):,}"
)

print(
    f"Songs                  : "
    f"{len(prepared.index_to_song_id):,}"
)

print(
    f"Train positives        : "
    f"{len(prepared.train_interactions):,}"
)

print(
    f"Validation positives   : "
    f"{len(prepared.validation_interactions):,}"
)

print(
    f"User feature matrix    : "
    f"{prepared.user_features.shape}"
)

print(
    f"Song metadata matrix   : "
    f"{prepared.song_metadata.shape}"
)

print(
    f"Lyric embedding matrix : "
    f"{prepared.lyric_embeddings.shape}"
)

print("=" * 70)
print("✅ HYBRID DATA PIPELINE COMPLETE")
print("=" * 70)

✅ Project directory: /Users/suvratsmacbookair/MyLocalFiles/LyricLens
✅ Data directory:    /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data
✅ Source directory:  /Users/suvratsmacbookair/MyLocalFiles/LyricLens/src
✅ interactions.csv
✅ user_features.csv
✅ song_characteristics.csv
✅ lyric_embeddings.csv
✅ final_songs_FIXED.csv

PROJECT MODULES
data_loader.py   → /Users/suvratsmacbookair/MyLocalFiles/LyricLens/src/data/data_loader.py
preprocess.py    → /Users/suvratsmacbookair/MyLocalFiles/LyricLens/src/evaluation/preprocess.py
emotion_theme.py → /Users/suvratsmacbookair/MyLocalFiles/LyricLens/src/features/emotion_theme.py

✅ Project modules imported successfully.

Loaded modules:
data_loader → /Users/suvratsmacbookair/MyLocalFiles/LyricLens/src/data/data_loader.py
preprocess  → /Users/suvratsmacbookair/MyLocalFiles/LyricLens/src/evaluation/preprocess.py

INTERACTIONS DATASET
Shape: (147000, 5)
Columns: ['user_id', 'song_id', 'label', 'is_hard_neg', 'genre']
⚠️ No interaction-strength c

## 3. Train Model

In [28]:
# =========================================================
# Training — Hybrid Recommendation Model
# =========================================================

from pathlib import Path
import sys


# ---------------------------------------------------------
# 1. Project paths
# ---------------------------------------------------------

PROJECT_DIR = Path(
    "/Users/suvratsmacbookair/MyLocalFiles/LyricLens"
)

DATA_DIR = PROJECT_DIR / "data"
SRC_DIR = PROJECT_DIR / "src"

TRAINING_DIR = SRC_DIR / "training"
MODELS_DIR = SRC_DIR / "models"

OUTPUT_DIR = PROJECT_DIR / "artifacts" / "training"


# ---------------------------------------------------------
# 2. Verify required directories/files
# ---------------------------------------------------------

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"❌ Project directory not found:\n{PROJECT_DIR}"
    )

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"❌ Data directory not found:\n{DATA_DIR}"
    )

if not TRAINING_DIR.exists():
    raise FileNotFoundError(
        f"❌ Training directory not found:\n{TRAINING_DIR}"
    )

if not MODELS_DIR.exists():
    raise FileNotFoundError(
        f"❌ Models directory not found:\n{MODELS_DIR}"
    )


trainer_file = TRAINING_DIR / "trainer.py"
hybrid_model_file = MODELS_DIR / "hybrid_model.py"

if not trainer_file.exists():
    raise FileNotFoundError(
        f"❌ trainer.py not found:\n{trainer_file}"
    )

if not hybrid_model_file.exists():
    raise FileNotFoundError(
        f"❌ hybrid_model.py not found:\n{hybrid_model_file}"
    )


print("✅ Project directory:")
print(PROJECT_DIR)

print("\n✅ Training directory:")
print(TRAINING_DIR)

print("\n✅ Models directory:")
print(MODELS_DIR)

print("\n✅ trainer.py:")
print(trainer_file)

print("\n✅ hybrid_model.py:")
print(hybrid_model_file)


# ---------------------------------------------------------
# 3. Add ALL relevant source directories to sys.path
# ---------------------------------------------------------

paths_to_add = [
    str(SRC_DIR),
    str(TRAINING_DIR),
    str(MODELS_DIR),
]


for path in paths_to_add:
    if path not in sys.path:
        sys.path.insert(0, path)


# ---------------------------------------------------------
# 4. Clear stale project modules
# ---------------------------------------------------------

modules_to_clear = [
    "trainer",
    "hybrid_model",
    "data_loader",
    "preprocess",
    "emotion_theme",
]

for module_name in modules_to_clear:
    if module_name in sys.modules:
        del sys.modules[module_name]


# ---------------------------------------------------------
# 5. Import training module
# ---------------------------------------------------------

from trainer import (
    TrainingConfig,
    train_hybrid_model,
)

print("\n✅ trainer.py imported successfully.")


# ---------------------------------------------------------
# 6. Verify imported module locations
# ---------------------------------------------------------

import trainer
import hybrid_model

print("\n" + "=" * 70)
print("IMPORTED MODULE LOCATIONS")
print("=" * 70)

print(
    "trainer      →",
    trainer.__file__
)

print(
    "hybrid_model →",
    hybrid_model.__file__
)


# ---------------------------------------------------------
# 7. Create output directory
# ---------------------------------------------------------

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\n✅ Output directory:")
print(OUTPUT_DIR)


# ---------------------------------------------------------
# 8. Check prepared dataset
# ---------------------------------------------------------

if "prepared" not in globals():
    raise RuntimeError(
        "❌ 'prepared' is not available.\n\n"
        "Run the previous hybrid-data preparation cell first."
    )

print("\n✅ Prepared hybrid dataset found.")


# ---------------------------------------------------------
# 9. Training configuration
# ---------------------------------------------------------

config = TrainingConfig(
    data_dir=str(DATA_DIR),
    output_dir=str(OUTPUT_DIR),

    epochs=8,
    batch_size=128,

    latent_dim=64,
    hidden_dim=128,

    negatives_per_positive=15,

    learning_rate=5e-4,
    weight_decay=1e-5,

    lambda_point=1.0,
    lambda_rank=1.5,

    interaction_type="dot",

    ranking_eval_k=10,

    seed=42,
)


# ---------------------------------------------------------
# 10. Output paths
# ---------------------------------------------------------

model_path = OUTPUT_DIR / "hybrid_model.pt"

metrics_path = OUTPUT_DIR / "hybrid_training_metrics.json"


# ---------------------------------------------------------
# 11. Display configuration
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print(f"Data directory        : {config.data_dir}")
print(f"Output directory      : {config.output_dir}")
print(f"Epochs                : {config.epochs}")
print(f"Batch size            : {config.batch_size}")
print(f"Latent dimension      : {config.latent_dim}")
print(f"Hidden dimension      : {config.hidden_dim}")
print(f"Negatives / positive  : {config.negatives_per_positive}")
print(f"Learning rate         : {config.learning_rate}")
print(f"Weight decay          : {config.weight_decay}")
print(f"Point loss weight     : {config.lambda_point}")
print(f"Ranking loss weight   : {config.lambda_rank}")
print(f"Interaction type      : {config.interaction_type}")
print(f"Ranking evaluation K  : {config.ranking_eval_k}")
print(f"Seed                  : {config.seed}")


# ---------------------------------------------------------
# 12. Train hybrid model
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("STARTING HYBRID MODEL TRAINING")
print("=" * 70)

model, training_metrics = train_hybrid_model(
    prepared=prepared,
    config=config,
    model_path=model_path,
    metrics_path=metrics_path,
)


# ---------------------------------------------------------
# 13. Verify saved files
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(f"Saved model   : {model_path}")
print(f"Saved metrics : {metrics_path}")

print(
    f"\nModel file exists   : {model_path.exists()}"
)

print(
    f"Metrics file exists : {metrics_path.exists()}"
)

print("=" * 70)
print("✅ HYBRID MODEL TRAINING FINISHED")
print("=" * 70)

✅ Project directory:
/Users/suvratsmacbookair/MyLocalFiles/LyricLens

✅ Training directory:
/Users/suvratsmacbookair/MyLocalFiles/LyricLens/src/training

✅ Models directory:
/Users/suvratsmacbookair/MyLocalFiles/LyricLens/src/models

✅ trainer.py:
/Users/suvratsmacbookair/MyLocalFiles/LyricLens/src/training/trainer.py

✅ hybrid_model.py:
/Users/suvratsmacbookair/MyLocalFiles/LyricLens/src/models/hybrid_model.py

✅ trainer.py imported successfully.

IMPORTED MODULE LOCATIONS
trainer      → /Users/suvratsmacbookair/MyLocalFiles/LyricLens/src/training/trainer.py
hybrid_model → /Users/suvratsmacbookair/MyLocalFiles/LyricLens/src/models/hybrid_model.py

✅ Output directory:
/Users/suvratsmacbookair/MyLocalFiles/LyricLens/artifacts/training

✅ Prepared hybrid dataset found.

TRAINING CONFIGURATION
Data directory        : /Users/suvratsmacbookair/MyLocalFiles/LyricLens/data
Output directory      : /Users/suvratsmacbookair/MyLocalFiles/LyricLens/artifacts/training
Epochs                : 8
Batc

## 4. Evaluate Recall@K and NDCG@K

In [30]:
from evaluate import evaluate_ranking

results = evaluate_ranking(
    model_path=model_path,
    output_dir=OUTPUT_DIR,
    k_values=[5, 10, 20],
)

results["metrics"]

{'@5': {'recall': 0.052296,
  'ndcg': 0.869185,
  'evaluated_users': 350,
  'n_candidates': 100,
  'protocol': 'sampled'},
 '@10': {'recall': 0.104525,
  'ndcg': 0.865371,
  'evaluated_users': 350,
  'n_candidates': 100,
  'protocol': 'sampled'},
 '@20': {'recall': 0.206235,
  'ndcg': 0.861533,
  'evaluated_users': 350,
  'n_candidates': 100,
  'protocol': 'sampled'}}

## 5. Generate Recommendations for One User

In [32]:
from hybrid_model import load_hybrid_checkpoint
from preprocess import load_prepared_hybrid_artifacts, build_positive_lookup

loaded_model, checkpoint = load_hybrid_checkpoint(model_path)
artifacts = load_prepared_hybrid_artifacts(OUTPUT_DIR)

user_idx = 0
scores = loaded_model.score_all_songs(
    user_id=user_idx,
    user_features=artifacts["user_features"],
    song_metadata=artifacts["song_metadata"],
    lyric_embeddings=artifacts["lyric_embeddings"],
)

train_lookup = build_positive_lookup(artifacts["train_interactions"])
for seen_song_idx in train_lookup.get(user_idx, set()):
    scores[int(seen_song_idx)] = -np.inf

top_k = 10
top_song_indices = np.argsort(-scores)[:top_k]

catalog = artifacts["song_catalog"].copy()
recommendations = catalog[catalog["song_idx"].isin(top_song_indices)].copy()
recommendations["score"] = recommendations["song_idx"].map({int(i): float(scores[int(i)]) for i in top_song_indices})
recommendations = recommendations.sort_values("score", ascending=False)

display(recommendations)

,song_idx,song_id,name,artists,album_name,genre,playlist_subgenre,year,popularity,effective_popularity,...,norm_danceability,norm_energy,norm_loudness,norm_speechiness,norm_acousticness,norm_instrumentalness,norm_liveness,norm_valence,norm_tempo,score
11777,11777,5hEt6trafIYyUpMiwshTwu,This (feat. Shawn Stockman and Omari Hardwick),"[""Tank"", ""Omari Hardwick"", ""Shawn Stockman""]",ELEVATION,R&B,neo soul,2019,3,3.0,...,0.428633,0.402988,0.865249,0.031383,0.785141,0.000000,0.090448,0.491423,0.306221,1.000000
6646,6646,5s5PIZVe0vWiMHrSLshapl,You Got It All,"[""Chir Cataran""]",Bossa Best,Jazz,unknown,2016,28,28.0,...,0.707799,0.556991,0.874788,0.002553,0.833333,0.000025,0.192975,0.671039,0.333858,1.000000
17951,17951,0swPDzca4dt7k91YN4c5DX,Two for Flinching,"[""Overslept""]",With or Without,Rock,hard rock,2019,3,3.0,...,0.510102,0.698994,0.843519,0.018085,0.149598,0.000000,0.356411,0.395560,0.435809,1.000000
26600,26600,1tnpSY3lWRLnRU0lr0e5BU,All the pretty little horses,"[""Traditional"", ""Richard Arame"", ""Shannon Call...",Lullabies of the World (LIEDERPROJEKT),Pop,indie poptimism,2013,9,9.0,...,0.552466,0.121982,0.611393,0.010957,0.956827,0.012864,0.113796,0.234107,0.460635,1.000000
2180,2180,60nqbqfajEWqh59fqVCvIf,Say You're Sorry,"[""Ministry""]",With Sympathy,Rock,hard rock,2006,29,29.0,...,0.554638,0.542991,0.729005,0.007340,0.050602,0.000002,0.100599,0.802220,0.489818,1.000000
2530,2530,6VlCcKXNssGfWEvCKpppdT,Know You've Got to Run - Demo,"[""Stephen Stills""]",Just Roll Tape - April 26th 1968,Folk,unknown,2007,11,11.0,...,0.620899,0.440989,0.783654,0.018085,0.683735,0.000010,0.107705,0.862765,0.226389,0.999999
18789,18789,6vjiNNOIO1DMAgF81FuwBF,Son Of A Widow,"[""mewithoutYou""]",Catch For Us The Foxes,Rock,hard rock,2004,25,25.0,...,0.376494,0.444989,0.782116,0.007766,0.508032,0.100201,0.072988,0.228052,0.269070,0.999999
26435,26435,0vjpfpch27QUf1cUs4seDl,down bad,"[""Yot Club""]",off the grid,Pop,indie poptimism,2022,51,51.0,...,0.453617,0.847997,0.819114,0.016064,0.000245,0.618090,0.080601,0.372351,0.774419,0.999999
8796,8796,1hgmxT8ypfN7v3AIf7elkE,DoZo - Live,"[""Puscifer""]",What Is... (Live),Rock,hard rock,2013,0,0.0,...,0.555725,0.777995,0.788402,0.017553,0.079217,0.026834,0.682266,0.382442,0.277944,0.999999
17204,17204,0LyZnEqqPwQvvNjCYOV6p7,"I Don't Need Love, I've Got My Band","[""The Radio Dept.""]","I Don't Need Love, I've Got My Band",Pop,indie poptimism,2019,35,35.0,...,0.315664,0.954999,0.876972,0.050745,0.014056,0.707538,0.100599,0.485368,0.639160,0.999999


In [36]:
recommendations[['song_id','genre','year','score']]

,song_id,genre,year,score
11777,5hEt6trafIYyUpMiwshTwu,R&B,2019,1.000000
6646,5s5PIZVe0vWiMHrSLshapl,Jazz,2016,1.000000
17951,0swPDzca4dt7k91YN4c5DX,Rock,2019,1.000000
26600,1tnpSY3lWRLnRU0lr0e5BU,Pop,2013,1.000000
2180,60nqbqfajEWqh59fqVCvIf,Rock,2006,1.000000
2530,6VlCcKXNssGfWEvCKpppdT,Folk,2007,0.999999
18789,6vjiNNOIO1DMAgF81FuwBF,Rock,2004,0.999999
26435,0vjpfpch27QUf1cUs4seDl,Pop,2022,0.999999
8796,1hgmxT8ypfN7v3AIf7elkE,Rock,2013,0.999999
17204,0LyZnEqqPwQvvNjCYOV6p7,Pop,2019,0.999999


In [3]:
import sys
from pathlib import Path

PROJECT_DIR = Path(
    "/Users/suvratsmacbookair/MyLocalFiles/LyricLens"
)

# Add all relevant src directories
for folder in [
    PROJECT_DIR,
    PROJECT_DIR / "src",
    PROJECT_DIR / "src" / "models",
    PROJECT_DIR / "src" / "evaluation",
    PROJECT_DIR / "src" / "data",
    PROJECT_DIR / "src" / "features",
]:
    folder = str(folder)
    if folder not in sys.path:
        sys.path.insert(0, folder)

print("Python paths fixed.")

Python paths fixed.


In [5]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    "/Users/suvratsmacbookair/MyLocalFiles/LyricLens"
)

sys.path.insert(0, str(PROJECT_DIR))

from recommender import HybridRecommender

df = pd.read_csv(
    PROJECT_DIR / "data" / "final_songs_FIXED.csv"
)

rec = HybridRecommender(
    df,
    PROJECT_DIR / "artifacts" / "training"
)

print("=" * 70)
print("HYBRID RECOMMENDER TEST")
print("=" * 70)

print("Backend:", rec.backend)
print("Songs:", len(rec.df))

nirvana = rec.search("Nirvana")

print("\nNirvana results:")
print(nirvana[["song_name", "artist"]].head(10))

idx = nirvana.index[0]

print("\nSelected song:")
print(df.loc[idx, ["song_name", "artist"]])

print("\nTop 5 recommendations:")
print(
    rec.recommend(
        idx,
        top_n=5
    )[["song_name", "artist"]]
)

HYBRID RECOMMENDER TEST
Backend: hybrid
Songs: 30000

Nirvana results:
                                               song_name  \
745                         Verse Chorus Verse - Outtake   
3527                                             Lithium   
6707                             Smells Like Teen Spirit   
6768                              Love Buzz - Remastered   
7308   The Money Will Roll Right In - 1992/Live at Re...   
8460                                           Swap Meet   
9886                         Nirvana (feat. pH-1 & WDBZ)   
11935                                              Polly   
12042                                    Come As You Are   
12746                                Smells Like Nirvana   

                               artist  
745                       ["Nirvana"]  
3527                      ["Nirvana"]  
6707                      ["Nirvana"]  
6768                      ["Nirvana"]  
7308                      ["Nirvana"]  
8460                      ["